In [ ]:
import torch
import torchaudio
import librosa
import soundfile as sf
import io
import numpy as np
from pydub import AudioSegment

c:\Users\mxlov\anaconda3\envs\speech-recog-go\Lib\site-packages\pydub\utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)


## Variable

In [2]:
AUDIO_PATH = './speech.wav'
bundle = torchaudio.pipelines.WAV2VEC2_ASR_BASE_960H
model = bundle.get_model()

## Transcriber / Predictor

In [3]:
def Transcriber(audiopath):
    waveform, sample_rate = torchaudio.load(audiopath)

    if sample_rate != bundle.sample_rate:
        waveform = torchaudio.functional.resample(waveform, sample_rate, bundle.sample_rate)

    with torch.inference_mode():
        emission, _ = model(waveform)

    class GreedyCTCDecoder (torch.nn.Module):
        def __init__(self, labels):
            super().__init__()
            self.labels = labels
            self.blank = 0

        def forward(self, emission: torch.Tensor):
            idx = torch.argmax(emission, dim=-1)
            idx = torch.unique_consecutive(idx, dim=-1)
            idx = [i for i in idx if i != self.blank]
            return "".join([self.labels[i] for i in idx])


    decoder = GreedyCTCDecoder(bundle.get_labels())
    txt = decoder(emission[0])
    txt = txt.replace("|", " ")
    return txt

## Support Function

In [ ]:
def ExportMenu (audio: AudioSegment):
    filename = None

    while True:
        print("Input Audio File (.wav): ")
        filename = input()
        
        if filename.endswith('.wav'):
            break
        else:
            print("Invalid Audio Type")

    audio.export(filename, "wav")
    return audio

def adj_volume (audio: AudioSegment, db):
    return audio + db

def adj_pitch (audio: AudioSegment, pitch):
    buffer = io.BytesIO()
    audio.export(buffer, format="wav")
    buffer.seek(0)

    signal, sample_rate = librosa.load(buffer, sr=None, mono=False)
    signal = librosa.effects.pitch_shift(signal, sr=sample_rate, n_steps=pitch)

    buffer = io.BytesIO()
    sf.write(buffer, signal, samplerate=sample_rate, format="wav")
    buffer.seek(0)

    audio = AudioSegment.from_wav(buffer)
    return audio

def add_whitenoise(audio: AudioSegment, noice_factor):
    buffer = io.BytesIO()
    audio.export(buffer, format="wav")
    buffer.seek(0)
    
    signal, sample_rate = librosa.load(buffer, sr=None, mono=False)
    noice = np.random.normal(0, signal.std(), signal.size)
    noice = signal + noice * noice_factor

    buffer = io.BytesIO()
    sf.write(buffer, noice, sample_rate)
    buffer.seek(0)

    audio = AudioSegment.from_wav(buffer)
    return audio

def add_repeat (audio: AudioSegment, rp):
    return audio * (rp + 1) 


def add_fadein (audio: AudioSegment, dur):
    return audio.fade_in(dur)

## Menu

In [ ]:
def Menu ():
    audio = AudioSegment.from_wav(AUDIO_PATH)

    while True:
        print("1. Pitch shift (Increase or decrease pitch)")
        print("2. Add white noise")
        print("3. Repeat clip")
        print("4. Insert fade in")
        print("5. Back to main menu")
        print(">> ")
        cc = input('')

        if cc == 1:
            while True:
                print("Input Pitch (-5 - 5): ")
                val = input()

                if val < -5 or val > 5:
                    print("Invalid Pitch Range")
                else:
                    break

            audio = adj_pitch(audio, val)
            ExportMenu(audio)
        
        elif cc == 2:
            while True:
                print("White Noise Level (0.1 - 0.5): ")
                val = input()

                if val < 0.1 or val > 0.5:
                    print("Invalid Noice Range")
                else:
                    break

            audio = add_whitenoise(audio, val)
            ExportMenu(audio)
        
        elif cc == 3:
            while True:
                print("Repeating Count (0<): ")
                val = input()

                if val < 0:
                    print("Invalid Repeating Count")
                else:
                    break

            audio = add_repeat(audio, val)
            ExportMenu(audio)
        
        elif cc == 4:
            while True:
                print("Fade In Duration (0<): ")
                val = input()

                if val < 0:
                    print("Invalid Fade In Duration")
                else:
                    break

            audio = add_fadein(audio, val)
            ExportMenu(audio)

        elif cc == 5:
            break

        else:
            print("Invalid Input")

In [ ]:
def MainMenu ():
    while True:
        print("1. Audio Augmentation")
        print("2. Predict / Transcribe Audio")
        print("3. Exit")
        print(">> ")
        cc = input('')

        if cc == 1:
            Menu()

        elif cc == 2:
            txt = Transcriber(AUDIO_PATH)
            print(txt)
        
        elif cc == 3:
            break

        else:
            print("Invalid Input")